Ejercicio 11. Calcular el costo del marketing por reservas canceladas

In [0]:
%sql
SELECT
    COUNT(*) AS reservas_canceladas_ta_to,
    ROUND(COUNT(*) * 1.50, 2) AS costo_marketing_usd
FROM cursodataacademy.proyecto_hotel.hotel_bookings
WHERE distribution_channel = 'TA/TO'
  AND is_canceled = 1;

reservas_canceladas_ta_to,costo_marketing_usd
40152,60228.00


Ejercicio 12. Calcular el costo de las cancelaciones tardías

In [0]:
%sql
WITH reservas AS (
    SELECT
        *,
        DATE_ADD(
            TO_DATE(
                CONCAT('01-', arrival_date_month, '-', arrival_date_year),
                'dd-MMMM-yyyy'
            ),
            CAST(arrival_date_day_of_month - 1 AS INT)
        ) AS arrival_date,
        TO_DATE(reservation_status_date, 'd/M/yy') AS cancellation_date
    FROM cursodataacademy.proyecto_hotel.hotel_bookings
),
cancelaciones_tardias AS (
    SELECT *
    FROM reservas
    WHERE is_canceled = 1
      AND DATEDIFF(arrival_date, cancellation_date) BETWEEN 0 AND 2
)
SELECT
    COUNT(*) AS cantidad_cancelaciones_tardias,
    COUNT(*) * 120 AS costo_operativo_usd
FROM cancelaciones_tardias;

cantidad_cancelaciones_tardias,costo_operativo_usd
3204,384480


El cálculo identifica las cancelaciones realizadas entre cero y dos días antes de la llegada y estima el costo operativo asociado, utilizando USD 120 por cada caso.

Ejercicio 13. LEFT JOIN entre reservas y países

In [0]:
%sql
WITH tb_reservas AS (
    SELECT * FROM VALUES
        ('PRT', 1),
        ('ESP', 0),
        ('FRA', 1),
        ('DEU', 0),
        ('ITA', 1)
    AS R(country, is_canceled)
),
tb_paises AS (
    SELECT * FROM VALUES
        ('PRT', 'Portugal'),
        ('ESP', 'España'),
        ('FRA', 'Francia'),
        ('DEU', 'Alemania')
    AS P(cod_pais, nombre_pais)
)
SELECT
    R.is_canceled,
    P.nombre_pais
FROM tb_reservas R
LEFT JOIN tb_paises P
    ON R.country = P.cod_pais;

is_canceled,nombre_pais
1,Portugal
0,España
1,Francia
0,Alemania
1,null


Ejercicio 14. TOP 5 de países con mayor cantidad de cancelaciones

In [0]:
%sql
WITH tb_reservas AS (
    SELECT * FROM VALUES
        ('PRT', 1),
        ('PRT', 1),
        ('ESP', 1),
        ('ESP', 0),
        ('FRA', 1),
        ('DEU', 1),
        ('ITA', 1),
        ('ITA', 1),
        ('ITA', 0),
        ('FRA', 1)
    AS R(country, is_canceled)
),
tb_paises AS (
    SELECT * FROM VALUES
        ('PRT', 'Portugal'),
        ('ESP', 'España'),
        ('FRA', 'Francia'),
        ('DEU', 'Alemania'),
        ('ITA', 'Italia')
    AS P(cod_pais, nombre_pais)
)
SELECT
    P.nombre_pais,
    COUNT(*) AS cantidad_cancelaciones
FROM tb_reservas R
INNER JOIN tb_paises P
    ON R.country = P.cod_pais
WHERE R.is_canceled = 1
GROUP BY P.nombre_pais
ORDER BY cantidad_cancelaciones DESC
LIMIT 5;

nombre_pais,cantidad_cancelaciones
Portugal,2
Francia,2
Italia,2
España,1
Alemania,1


Ejercicio 15. FULL OUTER JOIN – Relacionar reservas con vendedores

In [0]:
%sql
WITH tb_reservas AS (
    SELECT * FROM VALUES
        (1, 'TA/TO', 101),
        (2, 'Direct', 102),
        (3, 'Corporate', 103),
        (4, 'TA/TO', NULL)
    AS R(id_reserva, distribution_channel, id_vendedor)
),
tb_vendedores AS (
    SELECT * FROM VALUES
        (101, 'Carlos'),
        (102, 'María'),
        (104, 'Pedro')
    AS V(id_vendedor, nombre_vendedor)
)
SELECT
    R.distribution_channel,
    R.id_vendedor,
    V.nombre_vendedor,
    CASE
        WHEN R.id_reserva IS NULL THEN 'Vendedor sin reserva'
        WHEN V.id_vendedor IS NULL THEN 'Reserva sin vendedor'
        ELSE 'Reserva con vendedor'
    END AS estado_relacion
FROM tb_reservas R
FULL OUTER JOIN tb_vendedores V
    ON R.id_vendedor = V.id_vendedor;

distribution_channel,id_vendedor,nombre_vendedor,estado_relacion
TA/TO,101,Carlos,Reserva con vendedor
Direct,102,María,Reserva con vendedor
Corporate,103,null,Reserva sin vendedor
TA/TO,null,null,Reserva sin vendedor
null,null,Pedro,Vendedor sin reserva
